## Assignment 3

---

* **Course Instructor**

In [ ]:
import os, sys, random, warnings, math, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
import glob
from pathlib import Path
from tqdm.auto import tqdm
from torch.cuda.amp import autocast, GradScaler
from copy import deepcopy
import scipy.linalg
warnings.filterwarnings('ignore')

In [ ]:
SEED = 313
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_gpus = torch.cuda.device_count()
print(f"Device : {device}   |   GPUs : {num_gpus}")
for i in range(num_gpus):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name}  |  VRAM: {p.total_memory/1e9:.1f} GB")

# Con

In [ ]:
LATENT_DIM   = 128
IMAGE_SIZE   = 64
N_CHANNELS = 3
FEATURE_G    = 64       # generator base feature maps
FEATURE_D    = 64       # discriminator base feature maps
BATCH_SIZE   = 128
LR           = 4e-4
BETAS        = (0.5, 0.999)
EPOCHS_DCGAN = 70
EPOCHS = 50
CRITIC_STEPS = 3        # WGAN-GP critic updates per generator step
GP_LAMBDA    = 5       # gradient penalty coefficient
EMA_DECAY    = 0.999
NUM_WORKERS  = 4
SAVE_DIR     = Path("/kaggle/working/gan_checkpoints")
SAVE_DIR.mkdir(exist_ok=True)

# Data Prepration


In [ ]:
class FlatImageDataset(Dataset):
    EXT = {'.jpg', '.jpeg', '.png', '.webp'}

    def __init__(self, root, transform, max_samples=None):
        paths = []
        for e in self.EXT:
            paths += glob.glob(os.path.join(root, f'**/*{e}'), recursive=True)
        random.shuffle(paths)
        paths = paths[:max_samples] if max_samples else paths
        
        # --- THE SPEED CHANGE ---
        print(f"  Pre-loading {len(paths)} images into RAM for max speed...")
        self.samples = []
        for p in tqdm(paths, desc="Loading to RAM"):
            # Load and convert to tensor immediately
            img = transform(Image.open(p).convert('RGB'))
            self.samples.append(img)
            
        # Stack into one giant tensor [N, 3, 64, 64]
        self.data = torch.stack(self.samples)
        print(f"  Dataset ready. Memory used: {self.data.element_size() * self.data.nelement() / 1e6:.1f} MB")

    def __len__(self): 
        return len(self.data)

    def __getitem__(self, i):
        # No disk access here anymore!
        return self.data[i]
 
 
transform_64 = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

In [ ]:
ANIME_ROOT   = '/kaggle/input/datasets/soumikrakshit/anime-faces/data'
POKEMON_ROOT = '/kaggle/input/datasets/jackemartin/pokemon-sprites/pokemon_images/pokemondb.net'
 


Creating Dataset

In [ ]:
anime_dataset   = FlatImageDataset(ANIME_ROOT,   transform_64)
# pokemon_dataset = FlatImageDataset(POKEMON_ROOT, transform_64)
 
anime_loader   = DataLoader(anime_dataset,   BATCH_SIZE, shuffle=True,
                            num_workers=NUM_WORKERS, drop_last=True)
# pokemon_loader = DataLoader(pokemon_dataset, BATCH_SIZE, shuffle=True,
#                             num_workers=NUM_WORKERS, drop_last=True)
 
print(f"Anime batches: {len(anime_loader)} ")

**Display sample images from the dataset**

In [ ]:
def show_real_samples(loader, title, n=32):
    batch = next(iter(loader))[:n]
    grid = vutils.make_grid(batch, nrow=8, normalize=True, padding=2)
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.imshow(grid.permute(1, 2, 0).numpy())
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(f"/kaggle/working/{title.replace(' ', '_')}_real.png", dpi=150)
    plt.show()
 
show_real_samples(anime_loader,   'Anime Faces Dataset')


In [ ]:
#show_real_samples(pokemon_loader, 'Pokemon Sprites Images')

## 2 · Model Architectures

# DCGAN Model Architecture

In [ ]:
class DCGenerator(nn.Module):
    """DCGAN generator: z (128-d) → 64×64 image."""
    def __init__(self, latent=LATENT_DIM, fm=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent, fm*16, 4, 1, 0, bias=False),
            nn.BatchNorm2d(fm*16), nn.ReLU(True),               # 4
            nn.ConvTranspose2d(fm*16, fm*8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(fm*8),  nn.ReLU(True),               # 8
            nn.ConvTranspose2d(fm*8,  fm*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(fm*4),  nn.ReLU(True),               # 16
            nn.ConvTranspose2d(fm*4,  fm*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(fm*2),  nn.ReLU(True),               # 32
            nn.ConvTranspose2d(fm*2,  fm,   4, 2, 1, bias=False),
            nn.BatchNorm2d(fm),    nn.ReLU(True),               # 64-ish via extra conv
            nn.Conv2d(fm, N_CHANNELS, 3, 1, 1),
            nn.Tanh()
        )
        self._init()
 
    def _init(self):
        for m in self.modules():
            if isinstance(m, (nn.ConvTranspose2d, nn.Conv2d)):
                nn.init.normal_(m.weight, 0.0, 0.02)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.normal_(m.weight, 1.0, 0.02); nn.init.zeros_(m.bias)
 
    def forward(self, z):
        return self.net(z.view(-1, LATENT_DIM, 1, 1))
 
 
class DCDiscriminator(nn.Module):
    """DCGAN discriminator with spectral norm + label smoothing → sigmoid output."""
    def __init__(self, fm=64):
        super().__init__()
        SN = nn.utils.spectral_norm
        self.net = nn.Sequential(
            SN(nn.Conv2d(N_CHANNELS, fm,   4, 2, 1)),  nn.LeakyReLU(0.2, True),
            SN(nn.Conv2d(fm,   fm*2, 4, 2, 1)),  nn.BatchNorm2d(fm*2),  nn.LeakyReLU(0.2, True),
            SN(nn.Conv2d(fm*2, fm*4, 4, 2, 1)),  nn.BatchNorm2d(fm*4),  nn.LeakyReLU(0.2, True),
            SN(nn.Conv2d(fm*4, fm*8, 4, 2, 1)),  nn.BatchNorm2d(fm*8),  nn.LeakyReLU(0.2, True),
            SN(nn.Conv2d(fm*8, 1,    4, 1, 0)),
            #nn.Sigmoid() can't use this with mixed precision
        )
        self._init()
 
    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.normal_(m.weight, 0.0, 0.02)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.normal_(m.weight, 1.0, 0.02); nn.init.zeros_(m.bias)
 
    def forward(self, x): return self.net(x).view(-1)
 

# WGAN-GP Critic Architecture

In [ ]:
class WGANGenerator(nn.Module):

    def __init__(self, latent=LATENT_DIM, fm=128):
        super().__init__()
        self.project = nn.Sequential(
            nn.Linear(latent, fm*8 * 4 * 4, bias=False),
            nn.LeakyReLU(0.2, True)
        )
        self.body = nn.Sequential(
            self._block(fm*8, fm*4),    # 4→8
            self._block(fm*4, fm*2),    # 8→16
            self._block(fm*2, fm),      # 16→32
            self._block(fm,   fm//2),   # 32→64
        )
        self.out = nn.Sequential(
            nn.Conv2d(fm//2, N_CHANNELS, 3, 1, 1),
            nn.Tanh()
        )
        self._init()
 
    @staticmethod
    def _block(in_ch, out_ch):
        return nn.Sequential(
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(in_ch,  out_ch, 3, 1, 1, bias=False),
            nn.InstanceNorm2d(out_ch),
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False),
            nn.InstanceNorm2d(out_ch),
            nn.LeakyReLU(0.2, True),
        )
 
    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.normal_(m.weight, 0.0, 0.02)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0.0, 0.02)
 
    def forward(self, z):
        x = self.project(z).view(-1, 1024, 4, 4)
        return self.out(self.body(x))
 
 
class WGANCritic(nn.Module):
    
    def __init__(self, fm=64):
        super().__init__()
        SN = nn.utils.spectral_norm
        self.net = nn.Sequential(
            SN(nn.Conv2d(N_CHANNELS, fm,   4, 2, 1)),  nn.LeakyReLU(0.2, True),
            SN(nn.Conv2d(fm,   fm*2, 4, 2, 1)),  nn.InstanceNorm2d(fm*2),   nn.LeakyReLU(0.2, True),
            SN(nn.Conv2d(fm*2, fm*4, 4, 2, 1)),  nn.InstanceNorm2d(fm*4),   nn.LeakyReLU(0.2, True),
            SN(nn.Conv2d(fm*4, fm*8, 4, 2, 1)),  nn.InstanceNorm2d(fm*8),   nn.LeakyReLU(0.2, True),
            SN(nn.Conv2d(fm*8, 1,    4, 1, 0)),
            # NO sigmoid — raw critic score
        )
        self._init()
 
    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.normal_(m.weight, 0.0, 0.02)
 
    def forward(self, x): return self.net(x).view(-1)

**EMA helper**

In [ ]:
class EMA:
    def __init__(self, model, decay=EMA_DECAY):
        self.shadow = deepcopy(model).eval()
        self.decay  = decay
        for p in self.shadow.parameters(): p.requires_grad_(False)
 
    @torch.no_grad()
    def update(self, model):
        for s, m in zip(self.shadow.parameters(), model.parameters()):
            s.data = self.decay*s.data + (1-self.decay)*m.data
 
    @torch.no_grad()
    def __call__(self, z): return self.shadow(z)
 
 
def wrap(model):
    return nn.DataParallel(model) if num_gpus > 1 else model
 
G_dc = wrap(DCGenerator()).to(device)
D_dc = wrap(DCDiscriminator()).to(device)
G_wg = wrap(WGANGenerator()).to(device)
C_wg = wrap(WGANCritic()).to(device)

Moving to parallel gpus

In [ ]:
raw = lambda m: m.module if num_gpus > 1 else m
ema_dc = EMA(raw(G_dc)); ema_wg = EMA(raw(G_wg))

* Parametric count of each generator and discriminator

In [ ]:
print(f"DCGAN  G params : {sum(p.numel() for p in G_dc.parameters())/1e6:.2f}M")
print(f"DCGAN  D params : {sum(p.numel() for p in D_dc.parameters())/1e6:.2f}M")
print(f"WGAN-GP G params: {sum(p.numel() for p in G_wg.parameters())/1e6:.2f}M")
print(f"WGAN-GP C params: {sum(p.numel() for p in C_wg.parameters())/1e6:.2f}M")

Size check

In [ ]:
with torch.no_grad():
    z = torch.randn(4, LATENT_DIM).to(device)
    print(f"\nShape check — DCGAN G: {G_dc(z).shape} | WGAN G: {G_wg(z).shape}")
 

Clear memory

In [ ]:
import gc

def clear_vram():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

**Training Utilities**

In [ ]:
bce = nn.BCEWithLogitsLoss()
 
def dc_d_loss(real_scores, fake_scores):
    real_labels = torch.full_like(real_scores, 0.9)   # label smoothing
    fake_labels = torch.zeros_like(fake_scores)
    return bce(real_scores, real_labels) + bce(fake_scores, fake_labels)
 
def dc_g_loss(fake_scores):
    return bce(fake_scores, torch.ones_like(fake_scores))
 
def gradient_penalty(critic, real_imgs, fake_imgs):
    B     = real_imgs.size(0)
    alpha = torch.rand(B, 1, 1, 1, device=device)
    mix   = (alpha * real_imgs + (1-alpha) * fake_imgs).requires_grad_(True)
    score = critic(mix)
    grads = torch.autograd.grad(score, mix,
                                grad_outputs=torch.ones_like(score),
                                create_graph=True, retain_graph=True)[0]
    return ((grads.view(B, -1).norm(2, dim=1) - 1)**2).mean()
 
def noise(n): return torch.randn(n, LATENT_DIM, device=device)
 
fixed_z = noise(64)
 

In [ ]:
def display_gan_comparison(real_batch, fake_batch, epoch, model_name="GAN"):
    # Prepare grids (using top 16 images for clarity)
    real_grid = vutils.make_grid(real_batch[:16], nrow=4, normalize=True, padding=2)
    fake_grid = vutils.make_grid(fake_batch[:16], nrow=4, normalize=True, padding=2)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
    # Plot Real
    axes[0].imshow(real_grid.permute(1, 2, 0).cpu().numpy())
    axes[0].set_title(f"Real {model_name} Images", fontweight='bold')
    axes[0].axis('off')
    
    # Plot Fake
    axes[1].imshow(fake_grid.permute(1, 2, 0).cpu().numpy())
    axes[1].set_title(f"Generated (Epoch {epoch})", fontweight='bold')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()

## 4 · DCGAN Training

In [ ]:
scaler_G = torch.cuda.amp.GradScaler()
scaler_D = torch.cuda.amp.GradScaler()

# Note: Standard Adam is usually more stable for GANs than AdamW
opt_G_dc = torch.optim.Adam(G_dc.parameters(), lr=LR, betas=BETAS)
opt_D_dc = torch.optim.Adam(D_dc.parameters(), lr=LR*0.5, betas=BETAS)

history_dc = {'G': [], 'D': [], 'D_real': [], 'D_fake': []}
samples_dc = []

Resuming due to poor internet connect at hostel

In [ ]:
import os


RESUME_PATH = "/kaggle/input/models/alimusarizvi/dcgan/pytorch/default/1/dcgan_ep20.pth"
START_EPOCH = 1

if os.path.exists(RESUME_PATH):
    print(f"--- Resuming from checkpoint: {RESUME_PATH} ---")
    checkpoint = torch.load(RESUME_PATH, map_location=device)
    
    raw(G_dc).load_state_dict(checkpoint['G'])
    raw(D_dc).load_state_dict(checkpoint['D'])
    
    ema_dc.shadow.load_state_dict(raw(G_dc).state_dict())
    
    START_EPOCH = 21 
    print(f"Successfully loaded. Starting training from Epoch {START_EPOCH}.")
else:
    print("no such path")

Training

In [ ]:
for epoch in range(START_EPOCH, EPOCHS+1):
    G_dc.train(); D_dc.train()
    ep_G = ep_D = ep_Dr = ep_Df = 0.0

    for real in tqdm(anime_loader, desc=f"DC Ep {epoch:3d}/{EPOCHS}", leave=False):
        # Max speed with non_blocking
        real = real.to(device, non_blocking=True)
        B = real.size(0)

        # ── Discriminator (D) step ──
        opt_D_dc.zero_grad()
        with torch.cuda.amp.autocast():
            Dr = D_dc(real)
            fk = G_dc(noise(B)).detach()
            Df = D_dc(fk)
            # Ensure dc_d_loss uses BCEWithLogitsLoss
            ld = dc_d_loss(Dr, Df)
        
        # Use scaler_D here
        scaler_D.scale(ld).backward()
        scaler_D.step(opt_D_dc)
        scaler_D.update() # Update D scaler immediately

        # ── Generator (G) step ──
        opt_G_dc.zero_grad()
        with torch.cuda.amp.autocast():
            fk = G_dc(noise(B))
            lg = dc_g_loss(D_dc(fk))
            
        # Use scaler_G here
        scaler_G.scale(lg).backward()
        scaler_G.step(opt_G_dc)
        scaler_G.update() # Update G scaler immediately
        
        ema_dc.update(raw(G_dc))

        ep_G += lg.item(); ep_D += ld.item()
        # Apply sigmoid for logging since Discriminator now outputs raw logits
        ep_Dr += torch.sigmoid(Dr).mean().item()
        ep_Df += torch.sigmoid(Df).mean().item()

    # --- END OF EPOCH ---
    n = len(anime_loader)
    history_dc['G'].append(ep_G/n); history_dc['D'].append(ep_D/n)
    history_dc['D_real'].append(ep_Dr/n); history_dc['D_fake'].append(ep_Df/n)

    if epoch % 5 == 0:
        with torch.no_grad():
            sample_fakes = G_dc(noise(16)) 
            display_gan_comparison(real, sample_fakes, epoch, "DCGAN")
        clear_vram() # Memory management after display

    if epoch % 10 == 0:
        with torch.no_grad(): 
            s = ema_dc(fixed_z).cpu()
        samples_dc.append((epoch, s))
        torch.save({'G': raw(G_dc).state_dict(), 'D': raw(D_dc).state_dict()},
                   SAVE_DIR / f'dcgan_ep{epoch}.pth')
        print(f"  Ep {epoch}  G={history_dc['G'][-1]:.4f}  D={history_dc['D'][-1]:.4f}")

## 5 · WGAN-GP Training

In [ ]:
def gradient_penalty(critic, real_imgs, fake_imgs):
    B     = real_imgs.size(0)
    alpha = torch.rand(B, 1, 1, 1, device=device)
    mix   = (alpha * real_imgs + (1-alpha) * fake_imgs).requires_grad_(True)
    score = critic(mix)
    grads = torch.autograd.grad(score, mix,
                                grad_outputs=torch.ones_like(score),
                                create_graph=True, retain_graph=True)[0]
    
    # ADDED: + 1e-12 prevents NaN when the norm is exactly 0
    return ((grads.view(B, -1).norm(2, dim=1) + 1e-12 - 1)**2).mean()

In [ ]:
import gc

def get_raw(model):
    return model.module if isinstance(model, nn.DataParallel) else model

def clear_vram():
    gc.collect()
    torch.cuda.empty_cache()

# Separate Scalers (Crucial for fixing the AssertionError)
scaler_G_w = torch.cuda.amp.GradScaler()
scaler_C_w = torch.cuda.amp.GradScaler()

# Optimizers: RMSprop is highly recommended for WGAN stability
opt_G_wg = torch.optim.RMSprop(G_wg.parameters(), lr=5e-5)
opt_C_wg = torch.optim.RMSprop(C_wg.parameters(), lr=5e-5)

ema_G_wg = EMA(get_raw(G_wg))

history_wgan = {'G_loss': [], 'C_loss': [], 'GP': [], 'W_dist': []}
samples_wgan = []

**Training for WGAN-GP**

In [ ]:
def sample_noise(n):
    """Generates a batch of latent vectors from a normal distribution."""
    return torch.randn(n, LATENT_DIM, device=device)

fixed_noise = sample_noise(64)

In [ ]:
for epoch in range(1, EPOCHS + 1):
    G_wg.train(); C_wg.train()
    ep_G = ep_C = ep_gp = ep_wd = 0.0
    critic_count = 0
    pbar = tqdm(anime_loader, desc=f"Epoch {epoch:3d}/{EPOCHS}", leave=False)

    for real_imgs in pbar:
        real_imgs = real_imgs.to(device, non_blocking=True)
        B = real_imgs.size(0)

        # ── Critic update ──
        for _ in range(CRITIC_STEPS):
            z = sample_noise(B)
            
            with torch.cuda.amp.autocast():
                fake_imgs  = G_wg(z).detach() 
                score_real = C_wg(real_imgs)
                score_fake = C_wg(fake_imgs)
                
            # CRITICAL FIX: Disable autocast for Gradient Penalty
            # Calculate GP in pure float32 for numerical stability
            with torch.cuda.amp.autocast(enabled=False):
                gp = gradient_penalty(get_raw(C_wg), real_imgs.float(), fake_imgs.float())
                # Calculate final Critic loss in float32 as well
                loss_C = score_fake.float().mean() - score_real.float().mean() + GP_LAMBDA * gp
                w_distance = score_real.float().mean() - score_fake.float().mean()

            # set_to_none=True clears gradients entirely rather than setting to 0, saving VRAM
            opt_C_wg.zero_grad(set_to_none=True)
            scaler_C_w.scale(loss_C).backward()
            scaler_C_w.step(opt_C_wg)
            scaler_C_w.update()

            ep_C  += loss_C.item()
            ep_gp += gp.item()
            ep_wd += w_distance.item()
            critic_count += 1

        # ── Generator update ──
        z = sample_noise(B)
        with torch.cuda.amp.autocast():
            fake_imgs = G_wg(z)
            loss_G    = -C_wg(fake_imgs).mean()

        opt_G_wg.zero_grad(set_to_none=True)
        scaler_G_w.scale(loss_G).backward()
        scaler_G_w.step(opt_G_wg)
        scaler_G_w.update() 
        
        ema_G_wg.update(get_raw(G_wg))

        ep_G += loss_G.item()
        pbar.set_postfix(G=f"{loss_G.item():.3f}", W=f"{w_distance.item():.3f}")

    # Metrics logging
    n = len(anime_loader)
    history_wgan['G_loss'].append(ep_G / n)
    history_wgan['C_loss'].append(ep_C / critic_count)
    history_wgan['GP'].append(ep_gp / critic_count)
    history_wgan['W_dist'].append(ep_wd / critic_count)

    # Display comparison every 5 epochs
    if epoch % 2 == 0:
        with torch.no_grad():
            test_z = sample_noise(16)
            sample_fakes = G_wg(test_z)
            display_gan_comparison(real_imgs, sample_fakes, epoch, "WGAN-GP")
            
            sample = ema_G_wg(fixed_noise).cpu()
            samples_wgan.append((epoch, sample))
            torch.save({
            'G': get_raw(G_wg).state_dict(),
            'C': get_raw(C_wg).state_dict(),
            'epoch': epoch
        }, SAVE_DIR / f'wgan_ep{epoch}.pth')

        
        
    print(f"  Ep {epoch:3d} | G={history_wgan['G_loss'][-1]:.4f} | W={history_wgan['W_dist'][-1]:.4f}")
    
    # ── Memory Cleanup ──
    # Delete heavy tensors from the final batch to uncouple them from the graph
    del real_imgs, fake_imgs, loss_C, loss_G, score_real, score_fake, gp, z
    # Force garbage collection and clear GPU cache after every epoch
    clear_vram()

In [ ]:
import os
import gc

# 1. --- Define Checkpoint Loading Logic ---
# Set this to the specific checkpoint you want to resume from, or None to start fresh.
CHECKPOINT_PATH = '/kaggle/input/models/alimusarizvi/dcgan/pytorch/wganreal/1/wgan_ep12.pth' 
start_epoch = 1

if CHECKPOINT_PATH and os.path.exists(CHECKPOINT_PATH):
    print(f"Loading checkpoint from {CHECKPOINT_PATH}...")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    
    # Load weights into the RAW models to bypass DataParallel 'module.' prefix issues
    get_raw(G_wg).load_state_dict(checkpoint['G'])
    get_raw(C_wg).load_state_dict(checkpoint['C'])
    
    # It is highly recommended to load optimizer states if available, 
    # otherwise RMSprop loses its moving averages.
    if 'opt_G' in checkpoint and 'opt_C' in checkpoint:
        opt_G_wg.load_state_dict(checkpoint['opt_G'])
        opt_C_wg.load_state_dict(checkpoint['opt_C'])
        
    start_epoch = checkpoint['epoch'] + 1
    
    # Re-sync the EMA model with the newly loaded Generator weights
    ema_G_wg = EMA(get_raw(G_wg))
    
    print(f"Resuming training from epoch {start_epoch}")
else:
    print("No valid checkpoint found. Starting training from scratch.")


# 2. --- Updated Training Loop ---
# Notice range starts from start_epoch instead of 1
for epoch in range(start_epoch, EPOCHS + 1):
    G_wg.train(); C_wg.train()
    ep_G = ep_C = ep_gp = ep_wd = 0.0
    critic_count = 0
    pbar = tqdm(anime_loader, desc=f"Epoch {epoch:3d}/{EPOCHS}", leave=False)

    for real_imgs in pbar:
        real_imgs = real_imgs.to(device, non_blocking=True)
        B = real_imgs.size(0)

        # ── Critic update ──
        for _ in range(CRITIC_STEPS):
            z = sample_noise(B)
            
            with torch.cuda.amp.autocast():
                fake_imgs  = G_wg(z).detach() 
                score_real = C_wg(real_imgs)
                score_fake = C_wg(fake_imgs)
                
            # Disable autocast for Gradient Penalty
            with torch.cuda.amp.autocast(enabled=False):
                gp = gradient_penalty(get_raw(C_wg), real_imgs.float(), fake_imgs.float())
                loss_C = score_fake.float().mean() - score_real.float().mean() + GP_LAMBDA * gp
                w_distance = score_real.float().mean() - score_fake.float().mean()

            opt_C_wg.zero_grad(set_to_none=True)
            scaler_C_w.scale(loss_C).backward()
            scaler_C_w.step(opt_C_wg)
            scaler_C_w.update()

            ep_C  += loss_C.item()
            ep_gp += gp.item()
            ep_wd += w_distance.item()
            critic_count += 1

        # ── Generator update ──
        z = sample_noise(B)
        with torch.cuda.amp.autocast():
            fake_imgs = G_wg(z)
            loss_G    = -C_wg(fake_imgs).mean()

        opt_G_wg.zero_grad(set_to_none=True)
        scaler_G_w.scale(loss_G).backward()
        scaler_G_w.step(opt_G_wg)
        scaler_G_w.update() 
        
        ema_G_wg.update(get_raw(G_wg))

        ep_G += loss_G.item()
        pbar.set_postfix(G=f"{loss_G.item():.3f}", W=f"{w_distance.item():.3f}")

    # Metrics logging
    n = len(anime_loader)
    history_wgan['G_loss'].append(ep_G / n)
    history_wgan['C_loss'].append(ep_C / critic_count)
    history_wgan['GP'].append(ep_gp / critic_count)
    history_wgan['W_dist'].append(ep_wd / critic_count)

    # Display comparison & Save
    if epoch % 2 == 0:
        with torch.no_grad():
            test_z = sample_noise(16)
            sample_fakes = G_wg(test_z)
            display_gan_comparison(real_imgs, sample_fakes, epoch, "WGAN-GP")
            
            sample = ema_G_wg(fixed_noise).cpu()
            samples_wgan.append((epoch, sample))
            
        # UPGRADE: Saving optimizers is crucial for resuming RMSprop accurately
        torch.save({
            'G': get_raw(G_wg).state_dict(),
            'C': get_raw(C_wg).state_dict(),
            'opt_G': opt_G_wg.state_dict(),
            'opt_C': opt_C_wg.state_dict(),
            'epoch': epoch
        }, SAVE_DIR / f'wgan_ep{epoch}.pth')

    print(f"  Ep {epoch:3d} | G={history_wgan['G_loss'][-1]:.4f} | W={history_wgan['W_dist'][-1]:.4f}")
    
    # ── Memory Cleanup ──
    # 1. Uncouple specific heavy tensors
    try:
        del real_imgs, fake_imgs, loss_C, loss_G, score_real, score_fake, gp, z, test_z, sample_fakes, sample
    except NameError:
        pass # Catch cases where the loop breaks early or test tensors weren't generated
    
    # 2. Force CPU Garbage Collection (Cleans system RAM)
    gc.collect() 
    
    # 3. Clear GPU Cache (Cleans VRAM)
    torch.cuda.empty_cache()

## 6 · Training Loss Curves

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Training Metrics', fontsize=16, fontweight='bold')
 
ep_dc = range(1, EPOCHS_DCGAN + 1)
ep_wg = range(1, EPOCHS_WGAN + 1)
 
axes[0, 0].plot(ep_dc, history_dcgan['G_loss'], label='Generator', color='royalblue')
axes[0, 0].plot(ep_dc, history_dcgan['D_loss'], label='Discriminator', color='tomato')
axes[0, 0].set_title('DCGAN Loss'); axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('BCE Loss'); axes[0, 0].legend(); axes[0, 0].grid(alpha=0.3)
 
axes[0, 1].plot(ep_dc, history_dcgan['D_real'], label='D(real)', color='green')
axes[0, 1].plot(ep_dc, history_dcgan['D_fake'], label='D(fake)', color='orange')
axes[0, 1].axhline(0.5, linestyle='--', color='gray', alpha=0.5)
axes[0, 1].set_title('DCGAN Discriminator Outputs'); axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Score'); axes[0, 1].legend(); axes[0, 1].grid(alpha=0.3)
 
# Mode collapse indicator: variance of D outputs
d_variance = [abs(r - f) for r, f in
               zip(history_dcgan['D_real'], history_dcgan['D_fake'])]
axes[0, 2].plot(ep_dc, d_variance, color='purple')
axes[0, 2].set_title('DCGAN |D(real) − D(fake)|  (Mode Collapse Indicator)')
axes[0, 2].set_xlabel('Epoch'); axes[0, 2].set_ylabel('Gap'); axes[0, 2].grid(alpha=0.3)
 
axes[1, 0].plot(ep_wg, history_wgan['G_loss'], label='Generator', color='royalblue')
axes[1, 0].plot(ep_wg, history_wgan['C_loss'], label='Critic',    color='tomato')
axes[1, 0].set_title('WGAN-GP Loss'); axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Wasserstein Loss'); axes[1, 0].legend(); axes[1, 0].grid(alpha=0.3)
 
axes[1, 1].plot(ep_wg, history_wgan['W_dist'], color='teal')
axes[1, 1].set_title('WGAN-GP Wasserstein Distance'); axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('W-Distance'); axes[1, 1].grid(alpha=0.3)
 
axes[1, 2].plot(ep_wg, history_wgan['GP'], color='sienna')
axes[1, 2].set_title('WGAN-GP Gradient Penalty'); axes[1, 2].set_xlabel('Epoch')
axes[1, 2].set_ylabel('GP Value'); axes[1, 2].grid(alpha=0.3)
 
plt.tight_layout()
plt.savefig('/kaggle/working/training_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

## 7 · Generated Samples — Progressive Quality

In [ ]:
def plot_progression(samples_list, title):
    n = len(samples_list)
    fig, axes = plt.subplots(1, n, figsize=(n * 4, 4))
    if n == 1:
        axes = [axes]
    for ax, (ep, imgs) in zip(axes, samples_list):
        grid = vutils.make_grid(imgs[:16], nrow=4, normalize=True, padding=2)
        ax.imshow(grid.permute(1, 2, 0).numpy())
        ax.set_title(f'Epoch {ep}', fontsize=11)
        ax.axis('off')
    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    fname = title.replace(' ', '_').replace('/', '_')
    plt.savefig(f'/kaggle/working/{fname}.png', dpi=150, bbox_inches='tight')
    plt.show()
 
plot_progression(samples_dcgan, 'DCGAN — Sample Progression')

In [ ]:
plot_progression(samples_wgan,  'WGAN-GP — Sample Progression')

## 8 · Side-by-Side Final Comparison

In [ ]:
G_dcgan.eval(); G_wgan.eval()
with torch.no_grad():
    final_dc = ema_G_dc(fixed_noise[:32]).cpu()
    final_wg = ema_G_wg(fixed_noise[:32]).cpu()
 
fig, axes = plt.subplots(2, 1, figsize=(16, 9))
for ax, imgs, title in zip(axes,
                            [final_dc, final_wg],
                            ['DCGAN — 32 Generated Samples',
                             'WGAN-GP — 32 Generated Samples']):
    grid = vutils.make_grid(imgs, nrow=8, normalize=True, padding=2)
    ax.imshow(grid.permute(1, 2, 0).numpy())
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.axis('off')
 
plt.tight_layout()
plt.savefig('/kaggle/working/final_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 9 · FID Score Computation

Frechet Inception Distance


In [ ]:
class InceptionStats:
    """Compute Inception features for FID calculation."""
    def __init__(self):
        self.model = models.inception_v3(weights='DEFAULT', transform_input=False)
        self.model.fc = nn.Identity()
        self.model = self.model.to(device).eval()
 
    @torch.no_grad()
    def get_features(self, loader, n_samples=5000):
        features = []
        count = 0
        resize = transforms.Resize((299, 299))
        for imgs in loader:
            if count >= n_samples:
                break
            imgs = imgs.to(device)
            imgs = torch.stack([resize(img) for img in imgs])
            feats = self.model(imgs)
            features.append(feats.cpu().numpy())
            count += imgs.size(0)
        return np.concatenate(features, axis=0)[:n_samples]
 
    def stats(self, feats):
        mu = np.mean(feats, axis=0)
        sigma = np.cov(feats, rowvar=False)
        return mu, sigma
 
 
def compute_fid(mu1, sigma1, mu2, sigma2, eps=1e-6):
    diff = mu1 - mu2
    covmean, _ = scipy.linalg.sqrtm(sigma1 @ sigma2, disp=False)
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = diff @ diff + np.trace(sigma1 + sigma2 - 2 * covmean)
    return float(fid)
 
 


Computing FID score 

In [ ]:

inception = InceptionStats()
 
real_feats = inception.get_features(anime_loader)
mu_real, sigma_real = inception.stats(real_feats)
 
def fake_loader(generator, n=5000):
    imgs = []
    while len(imgs) < n:
        z = sample_noise(BATCH_SIZE)
        with torch.no_grad():
            batch = generator(z).cpu()
        imgs.append(batch)
    return torch.cat(imgs)[:n]
 
dcgan_imgs = fake_loader(G_dcgan)
wgan_imgs  = fake_loader(G_wgan)
 
dc_loader  = DataLoader(dcgan_imgs, batch_size=BATCH_SIZE)
wg_loader  = DataLoader(wgan_imgs,  batch_size=BATCH_SIZE)
 
dc_feats   = inception.get_features(dc_loader)
wg_feats   = inception.get_features(wg_loader)
 
mu_dc, sigma_dc = inception.stats(dc_feats)
mu_wg, sigma_wg = inception.stats(wg_feats)
 
fid_dcgan = compute_fid(mu_real, sigma_real, mu_dc, sigma_dc)
fid_wgan  = compute_fid(mu_real, sigma_real, mu_wg, sigma_wg)
 
print(f"\n{'='*40}")
print(f"  FID Score — DCGAN  : {fid_dcgan:.2f}")
print(f"  FID Score — WGAN-GP: {fid_wgan:.2f}")
print(f"  {'WGAN-GP' if fid_wgan < fid_dcgan else 'DCGAN'} achieves lower FID (better diversity)")
print(f"{'='*40}"

Graph displayed

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
models_names = ['DCGAN', 'WGAN-GP']
fid_scores   = [fid_dcgan, fid_wgan]
colors = ['#5b9bd5', '#ed7d31']
bars = ax.bar(models_names, fid_scores, color=colors, width=0.45, edgecolor='black')
for bar, score in zip(bars, fid_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{score:.2f}', ha='center', va='bottom', fontweight='bold', fontsize=13)
ax.set_title('FID Score Comparison (lower is better)', fontsize=14, fontweight='bold')
ax.set_ylabel('FID Score'); ax.set_ylim(0, max(fid_scores) * 1.2)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/fid_comparison.png', dpi=150)
plt.show()

In [ ]:
results = {
    'DCGAN_FID':  fid_dcgan,
    'WGAN_GP_FID': fid_wgan,
    'epochs_trained': {'DCGAN': EPOCHS_DCGAN, 'WGAN_GP': EPOCHS_WGAN}
}
with open('/kaggle/working/results.json', 'w') as f:
    json.dump(results, f, indent=2)

In [1]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file.endswith('.pth') or file.endswith('.pt'):
            print(os.path.join(root, file))